# ConvNeXt 224 + 320 and ResNet-152: probability ensemble

No training and no GPU required. The ConvNeXt branch reproduces the best current
submission (`submission_blend_224_320.csv`, public score **0.95039**): equal mean of
224 and 320 predictions **without TTA**, using thresholds fitted to their matching OOF mean.

The experimental ensemble has fixed weights **50% ConvNeXt blend + 50% ResNet**
(equivalently 25% ConvNeXt-224, 25% ConvNeXt-320, 50% ResNet). We do not mix binary
submissions, add `keep_one`, or search weights against the public leaderboard.

ResNet in `ResNet.ipynb` is a single fold-0 model, not a five-fold OOF ensemble.
Evaluation and ensemble threshold fitting therefore use ONLY its declared held-out
fold, aligned by image_id with the matching ConvNeXt OOF rows. All-five ConvNeXt
test predictions are averaged for the final submission. A larger training ensemble
at test time differs from single held-out-fold OOF predictions, as in our existing CV pipeline.

Attach both ConvNeXt backups (224 and 320, including prediction folders) and
the `resnet_for_ensemble` export from `resnet_export_for_ensemble.ipynb`.
Saved model weights themselves are not read here.

The explicit two-stage formula is also implemented in `ensemble_convnext_resnet.py`. This notebook stays standalone for Kaggle. ResNet weights/probabilities are still required; the supplied binary submission cannot run this probability pipeline. Historical hard voting is in `archive/experiments/`.

## Imports and paths
Default working paths are reused. Backups attached under Input are discovered when
there is one matching copy; set explicit paths if multiple versions are attached.

In [ ]:
import os
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from IPython.display import display, FileLink

BASE_CV_DIR = Path('/kaggle/working/convnext_cv')
FINE_CV_DIR = Path('/kaggle/working/convnext_cv_320_fast')
BASE_PREDICTIONS_DIR = Path('/kaggle/working/convnext_cv_ensemble')
FINE_PREDICTIONS_DIR = Path('/kaggle/working/convnext_cv_320_fast_ensemble')
RESNET_DIR = Path('/kaggle/working/resnet_for_ensemble')
DATA_DIR = Path('/kaggle/input/competitions/cats-and-dogs')
OUTPUT_DIR = Path('/kaggle/working/convnext_resnet_ensemble')
LABEL_COLS = ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']
CONVNEXT_WEIGHT = 0.5
RESNET_WEIGHT = 0.5
assert CONVNEXT_WEIGHT == RESNET_WEIGHT == 0.5, 'This experiment uses predeclared equal branch weights.'


## Validate saved predictions and the common held-out split
Probabilities, labels, duplicate groups, split fingerprints and source manifests must
match. A probability bundle for another split is rejected instead of evaluating on training rows.

In [ ]:
def read_unique(path):
    frame = pd.read_csv(path, dtype={'image_id': str})
    assert len(frame) and frame.image_id.notna().all() and frame.image_id.is_unique, str(path)
    return frame

def align(frame, ids):
    ids = list(ids)
    assert len(ids) == len(set(ids)) and set(frame.image_id) == set(ids), 'Image IDs differ.'
    return frame.set_index('image_id').loc[ids].reset_index()

def probabilities(frame, columns):
    values = frame[columns].to_numpy(dtype=np.float64)
    assert values.shape == (len(frame), len(LABEL_COLS)) and np.isfinite(values).all()
    assert ((values >= 0) & (values <= 1)).all(), 'Expected probabilities in [0, 1].'
    return values

def file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def folds_digest(frame):
    canonical = frame[['image_id', *LABEL_COLS, 'cluster', 'fold']].sort_values('image_id').copy()
    canonical[['cluster', 'fold', *LABEL_COLS]] = canonical[['cluster', 'fold', *LABEL_COLS]].astype('int64')
    return hashlib.sha256(canonical.to_csv(index=False, lineterminator=chr(10)).encode()).hexdigest()

def find_directories(required_files):
    found = []
    for root in [Path('/kaggle/working'), Path('/kaggle/input')]:
        if not root.is_dir():
            continue
        for directory, subdirs, files in os.walk(root):
            subdirs[:] = [name for name in subdirs if name not in {'train', 'test', 'images', 'wandb', '.virtual_documents'}]
            if set(required_files).issubset(files):
                found.append(Path(directory))
    return found

def resolve_directory(preferred, required_files, predicate=lambda directory: True):
    preferred = Path(preferred) if preferred is not None else None
    if preferred is not None and all((preferred / name).is_file() for name in required_files):
        assert predicate(preferred), f'Incompatible files in {preferred}'
        return preferred
    candidates = [directory for directory in find_directories(required_files) if predicate(directory)]
    if len(candidates) != 1:
        raise FileNotFoundError(
            f'Set the directory explicitly; attach the saved Output or upload the backup. '
            f'Required: {required_files}. Candidates: {candidates}'
        )
    return candidates[0]

def matches_cv(directory, size):
    manifest = json.loads((directory / 'ensemble_config.json').read_text())
    return (manifest.get('model_name') == 'convnext_tiny.fb_in1k'
            and manifest.get('preprocessing', {}).get('image_size') == size)

BASE_CV_DIR = resolve_directory(BASE_CV_DIR, ['ensemble_config.json', 'oof_predictions.csv'], lambda p: matches_cv(p, 224))
FINE_CV_DIR = resolve_directory(FINE_CV_DIR, ['ensemble_config.json', 'oof_predictions.csv'], lambda p: matches_cv(p, 320))
base_manifest = json.loads((BASE_CV_DIR / 'ensemble_config.json').read_text())
fine_manifest = json.loads((FINE_CV_DIR / 'ensemble_config.json').read_text())
for manifest in [base_manifest, fine_manifest]:
    assert manifest['label_cols'] == LABEL_COLS and manifest['folds'] == list(range(5))
    assert manifest['aggregation'] == 'equal_mean_sigmoid_probabilities'
assert base_manifest['folds_fingerprint'] == fine_manifest['folds_fingerprint']

def matches_predictions(directory, source_manifest):
    manifest = json.loads((directory / 'ensemble_config.json').read_text())
    return (manifest.get('cv_fingerprint') == source_manifest['cv_fingerprint']
            and manifest.get('checkpoint_hashes') == source_manifest['checkpoint_hashes'])

BASE_PREDICTIONS_DIR = resolve_directory(BASE_PREDICTIONS_DIR,
    ['ensemble_config.json', 'test_probabilities_original.csv'], lambda p: matches_predictions(p, base_manifest))
FINE_PREDICTIONS_DIR = resolve_directory(FINE_PREDICTIONS_DIR,
    ['ensemble_config.json', 'test_probabilities_original.csv'], lambda p: matches_predictions(p, fine_manifest))
RESNET_DIR = resolve_directory(RESNET_DIR, ['resnet_export.json', 'validation_predictions.csv', 'test_probabilities.csv'])
resnet_manifest = json.loads((RESNET_DIR / 'resnet_export.json').read_text())
assert resnet_manifest['schema'] == 'resnet_probability_export_v1'
assert resnet_manifest['model_name'] == 'resnet152.a1_in1k'
assert resnet_manifest['label_cols'] == LABEL_COLS
assert resnet_manifest['tta'] is False and resnet_manifest['postprocessing'] == 'none'
assert resnet_manifest['folds_fingerprint'] == base_manifest['folds_fingerprint'], 'ResNet used a different split.'
for filename in ['validation_predictions.csv', 'test_probabilities.csv']:
    assert file_digest(RESNET_DIR / filename) == resnet_manifest['file_hashes'][filename], f'Changed ResNet file: {filename}'
held_out_fold = int(resnet_manifest['held_out_fold'])
assert held_out_fold in range(5)

DATA_DIR = resolve_directory(DATA_DIR, ['train.csv', 'test.csv', 'sample_submission.csv'])
train = read_unique(DATA_DIR / 'train.csv')
base_oof = align(read_unique(BASE_CV_DIR / 'oof_predictions.csv'), train.image_id)
fine_oof = align(read_unique(FINE_CV_DIR / 'oof_predictions.csv'), train.image_id)
assert np.array_equal(base_oof[LABEL_COLS].to_numpy(), train[LABEL_COLS].to_numpy())
assert base_oof[LABEL_COLS].isin([0, 1]).all().all()
assert set(base_oof.fold.unique()) == set(range(5))
assert base_oof.groupby('cluster').fold.nunique().eq(1).all()
assert folds_digest(base_oof) == base_manifest['folds_fingerprint']
assert folds_digest(fine_oof) == fine_manifest['folds_fingerprint']
assert np.array_equal(base_oof[['fold', 'cluster', *LABEL_COLS]].to_numpy(),
                      fine_oof[['fold', 'cluster', *LABEL_COLS]].to_numpy())
convnext_oof = (probabilities(base_oof, [label + '_prob' for label in LABEL_COLS])
                + probabilities(fine_oof, [label + '_prob' for label in LABEL_COLS])) / 2

common_rows = base_oof.loc[base_oof.fold == held_out_fold].reset_index(drop=True)
resnet_val = align(read_unique(RESNET_DIR / 'validation_predictions.csv'), common_rows.image_id)
assert np.array_equal(resnet_val[['fold', 'cluster', *LABEL_COLS]].to_numpy(),
                      common_rows[['fold', 'cluster', *LABEL_COLS]].to_numpy()), 'ResNet held-out targets or groups differ.'
convnext_val = convnext_oof[base_oof.fold.to_numpy() == held_out_fold]
resnet_val_probs = probabilities(resnet_val, [label + '_prob' for label in LABEL_COLS])
ensemble_val = CONVNEXT_WEIGHT * convnext_val + RESNET_WEIGHT * resnet_val_probs
targets = common_rows[LABEL_COLS].to_numpy(dtype=np.int32)
print('Common held-out fold:', held_out_fold, '| rows:', len(common_rows))
print('Positives:', dict(zip(LABEL_COLS, targets.sum(axis=0).tolist())))
print('ResNet checkpoint:', resnet_manifest['checkpoint_filename'], resnet_manifest['checkpoint_sha256'][:12])


## Thresholds and validation comparison
The same 0.05–0.95 per-class grid is reused. The table compares all three branches
on the exact same held-out photos, at 0.5 and with thresholds fitted to those photos.
Fitted scores reuse calibration labels and are optimistic, especially for rare pets;
they are not independent CV estimates and do not guarantee a better Kaggle score.
The ConvNeXt reproduction separately retains thresholds fitted on ALL its OOF rows.

In [ ]:
THRESHOLD = 0.5

def tune_thresholds(targets, probabilities):
    thresholds = np.full(len(LABEL_COLS), THRESHOLD, dtype=np.float64)
    candidates = np.round(np.linspace(0.05, 0.95, 91), 2)
    for j, label in enumerate(LABEL_COLS):
        if np.unique(targets[:, j]).size < 2:
            print(f'{label}: keeping {THRESHOLD}; validation does not contain both outcomes.')
            continue
        scores = np.array([
            f1_score(targets[:, j], probabilities[:, j] >= t, zero_division=0)
            for t in candidates
        ])
        tied = np.flatnonzero(np.isclose(scores, scores.max(), rtol=0, atol=1e-12))
        best = tied[np.argmin(np.abs(candidates[tied] - THRESHOLD))]
        thresholds[j] = candidates[best]
    return thresholds
validation_candidates = {'convnext_224_320': convnext_val, 'resnet152': resnet_val_probs,
                         'convnext_resnet_50_50': ensemble_val}
cutoffs_by_candidate, metric_rows, label_rows = {}, [], []
for name, values in validation_candidates.items():
    cutoffs_by_candidate[name] = tune_thresholds(targets, values)
    for threshold_source, cutoffs in [('fixed_0_5', np.full(len(LABEL_COLS), 0.5)),
                                      ('fitted_common_validation', cutoffs_by_candidate[name])]:
        predicted = values >= cutoffs
        metric_rows.append({'candidate': name, 'threshold_source': threshold_source,
            'rows': len(targets), 'macro_f1': f1_score(targets, predicted, average='macro', zero_division=0),
            'val_accuracy': accuracy_score(targets, predicted)})
        for j, label in enumerate(LABEL_COLS):
            y, p = targets[:, j], predicted[:, j]
            label_rows.append({'candidate': name, 'threshold_source': threshold_source, 'label': label,
                'positives': int(y.sum()), 'f1': f1_score(y, p, zero_division=0),
                'precision': precision_score(y, p, zero_division=0), 'recall': recall_score(y, p, zero_division=0),
                'false_positives': int(((p == 1) & (y == 0)).sum()),
                'false_negatives': int(((p == 0) & (y == 1)).sum()), 'threshold': float(cutoffs[j])})
comparison, per_label = pd.DataFrame(metric_rows), pd.DataFrame(label_rows)
display(comparison)
display(per_label.loc[per_label.threshold_source == 'fitted_common_validation'])
convnext_full_oof_thresholds = tune_thresholds(base_oof[LABEL_COLS].to_numpy(), convnext_oof)
ensemble_thresholds = cutoffs_by_candidate['convnext_resnet_50_50']
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(OUTPUT_DIR / 'validation_comparison.csv', index=False)
per_label.to_csv(OUTPUT_DIR / 'validation_per_label.csv', index=False)
validation_export = common_rows[['image_id', 'cluster', 'fold', *LABEL_COLS]].copy()
for name, values in validation_candidates.items():
    for j, label in enumerate(LABEL_COLS):
        validation_export[f'{name}_{label}_prob'] = values[:, j]
validation_export.to_csv(OUTPUT_DIR / 'common_validation_predictions.csv', index=False)


## Test ensemble and submission
Rows are aligned by image_id and exported in official sample order. The primary
experiment is `submission_convnext_resnet.csv`. The unchanged ConvNeXt baseline
is also reproduced for comparison. No upload to the competition is performed.

In [ ]:
sample = read_unique(DATA_DIR / 'sample_submission.csv')
test = read_unique(DATA_DIR / 'test.csv')
assert set(sample.columns) == {'image_id', *LABEL_COLS}
assert set(sample.image_id) == set(test.image_id)
base_test = align(read_unique(BASE_PREDICTIONS_DIR / 'test_probabilities_original.csv'), sample.image_id)
fine_test = align(read_unique(FINE_PREDICTIONS_DIR / 'test_probabilities_original.csv'), sample.image_id)
resnet_test = align(read_unique(RESNET_DIR / 'test_probabilities.csv'), sample.image_id)
convnext_test = (probabilities(base_test, LABEL_COLS) + probabilities(fine_test, LABEL_COLS)) / 2
ensemble_test = CONVNEXT_WEIGHT * convnext_test + RESNET_WEIGHT * probabilities(resnet_test, LABEL_COLS)

def save_submission(filename, values, cutoffs):
    submission = sample.copy()
    submission[LABEL_COLS] = (values >= cutoffs).astype(np.int32)
    assert len(submission) == len(test) and not submission.isna().any().any()
    assert submission[LABEL_COLS].isin([0, 1]).all().all()
    assert submission.columns.tolist() == sample.columns.tolist()
    path = OUTPUT_DIR / filename
    submission.to_csv(path, index=False)
    print('Saved:', path, '| rows:', len(submission))
    display(FileLink(str(path)))

save_submission('submission_convnext_baseline.csv', convnext_test, convnext_full_oof_thresholds)
save_submission('submission_convnext_resnet.csv', ensemble_test, ensemble_thresholds)
for name, values in [('convnext', convnext_test), ('convnext_resnet', ensemble_test)]:
    frame = pd.DataFrame(values, columns=LABEL_COLS)
    frame.insert(0, 'image_id', sample.image_id.to_numpy())
    frame.to_csv(OUTPUT_DIR / f'test_probabilities_{name}.csv', index=False)

metadata = {
    'schema': 'convnext_resnet_ensemble_v1', 'label_cols': LABEL_COLS,
    'weights': {'convnext224': 0.25, 'convnext320': 0.25, 'resnet152': 0.5}, 'tta': False,
    'base_cv_fingerprint': base_manifest['cv_fingerprint'], 'fine_cv_fingerprint': fine_manifest['cv_fingerprint'],
    'base_checkpoint_hashes': base_manifest['checkpoint_hashes'], 'fine_checkpoint_hashes': fine_manifest['checkpoint_hashes'],
    'folds_fingerprint': base_manifest['folds_fingerprint'], 'resnet_export': resnet_manifest,
    'calibration_fold': held_out_fold, 'threshold_source': 'common held-out validation; fitted score is optimistic',
    'thresholds': {'convnext_baseline_all_oof': convnext_full_oof_thresholds.tolist(),
                   'convnext_resnet_common_validation': ensemble_thresholds.tolist()},
    'input_hashes': {
        'base_oof': file_digest(BASE_CV_DIR / 'oof_predictions.csv'),
        'fine_oof': file_digest(FINE_CV_DIR / 'oof_predictions.csv'),
        'base_test': file_digest(BASE_PREDICTIONS_DIR / 'test_probabilities_original.csv'),
        'fine_test': file_digest(FINE_PREDICTIONS_DIR / 'test_probabilities_original.csv')},
}
(OUTPUT_DIR / 'ensemble_config.json').write_text(json.dumps(metadata, indent=2))
print('Submit the experimental file: submission_convnext_resnet.csv')
